# Chapter 7: Performance and Optimization

**Duration: 3 hours**

## 7.1 Why Julia is Fast

Julia achieves C-like performance through:
- JIT compilation via LLVM
- Type specialization
- Inlining and SIMD vectorization
- No interpreter overhead for hot paths

## 7.2 Benchmarking with BenchmarkTools.jl

In [ ]:
using Pkg
Pkg.add("BenchmarkTools")

In [ ]:
using BenchmarkTools

# Wrong way to benchmark (includes compilation)
@time sum(rand(10^6))

# Right way — @benchmark runs many iterations
@benchmark sum(rand(10^6))

In [ ]:
# Compare implementations
function mysum_naive(v)
    s = 0.0
    for x in v
        s += x
    end
    return s
end

v = rand(10^6)
@btime mysum_naive($v)
@btime sum($v)

## 7.3 Type Stability

A function is *type-stable* if the return type can be inferred from the argument types alone. Type instability is the #1 performance killer in Julia.

In [ ]:
# Type UNSTABLE — bad!
function unstable_pos(x)
    if x > 0
        return x      # returns the same type as x
    else
        return 0      # returns Int! (not same type if x is Float64)
    end
end

# Type STABLE — good!
function stable_pos(x)
    if x > 0
        return x
    else
        return zero(x)  # returns 0 of the same type as x
    end
end

# Check with @code_warntype
@code_warntype unstable_pos(1.5)

In [ ]:
@code_warntype stable_pos(1.5)

In [ ]:
# Performance difference
v = randn(10^6)
@btime sum(unstable_pos.($v))
@btime sum(stable_pos.($v))

## 7.4 Avoiding Global Variables

In [ ]:
# BAD — global variable
x_global = rand(10^6)
function sum_global()
    s = 0.0
    for x in x_global
        s += x
    end
    return s
end

# GOOD — pass as argument
function sum_local(v)
    s = 0.0
    for x in v
        s += x
    end
    return s
end

# Or use const for globals
const X_CONST = rand(10^6)
function sum_const()
    s = 0.0
    for x in X_CONST
        s += x
    end
    return s
end

v = rand(10^6)
@btime sum_global()
@btime sum_local($v)
@btime sum_const()

## 7.5 Memory Allocation and Views

In [ ]:
using BenchmarkTools

A = rand(1000, 1000)

# BAD — allocates a new array
function col_sum_copy(A)
    s = 0.0
    for j in 1:size(A, 2)
        col = A[:, j]    # allocates!
        s += sum(col)
    end
    return s
end

# GOOD — use @view
function col_sum_view(A)
    s = 0.0
    for j in 1:size(A, 2)
        col = @view A[:, j]  # no allocation
        s += sum(col)
    end
    return s
end

@btime col_sum_copy($A)
@btime col_sum_view($A)

## 7.6 Column-Major Order

In [ ]:
# Julia stores matrices column-major (like Fortran)
A = rand(1000, 1000)

# BAD — row-major traversal
function sum_row_major(A)
    s = 0.0
    for i in 1:size(A, 1)    # rows
        for j in 1:size(A, 2)  # columns
            s += A[i, j]
        end
    end
    return s
end

# GOOD — column-major traversal
function sum_col_major(A)
    s = 0.0
    for j in 1:size(A, 2)    # columns (outer)
        for i in 1:size(A, 1)  # rows (inner)
            s += A[i, j]
        end
    end
    return s
end

@btime sum_row_major($A)
@btime sum_col_major($A)

## 7.7 Profiling

In [ ]:
# Using @profile
using Profile
using LinearAlgebra

function heavy_computation(n)
    A = rand(n, n)
    B = rand(n, n)
    C = A * B
    vals = eigvals(C)
    return sum(vals)
end

# Warm up
heavy_computation(10)

# Profile
# Profile.clear()
# @profile heavy_computation(500)
# Profile.print()

In [ ]:
# Allocation tracking with @allocated
function test_alloc(n)
    v = zeros(n)
    for i in 1:n
        v[i] = sin(i)
    end
    return v
end

test_alloc(10)  # warm up
println("Bytes allocated: $(@allocated test_alloc(10000))")

## 7.8 Performance Tips Summary

1. Write type-stable functions
2. Avoid global variables (use `const` or pass as arguments)
3. Pre-allocate output arrays
4. Use `@view` instead of slicing
5. Traverse arrays in column-major order
6. Use `@inbounds` in tight loops (after verifying correctness)
7. Fuse broadcasts with `@.`
8. Use concrete types in containers (`Vector{Float64}` not `Vector{Any}`)

In [ ]:
# @inbounds example (removes bounds checking)
function fast_sum(v)
    s = zero(eltype(v))
    @inbounds for i in eachindex(v)
        s += v[i]
    end
    return s
end

v = rand(10^6)
@btime fast_sum($v)
@btime sum($v)

## Exercises

In [ ]:
# Exercise 1: Write two versions of matrix-vector multiply
# (row-major vs column-major loop). Benchmark both on 1000×1000.
# Your code here

In [ ]:
# Exercise 2: Write a type-unstable function and fix it.
# Verify with @code_warntype.
# Your code here

In [ ]:
# Exercise 3: Compare A[:, j] vs @view A[:, j] in a loop
# over columns of a 10000×1000 matrix.
# Your code here

In [ ]:
# Exercise 4: Profile a function that does eigenvalue decomposition.
# Identify the bottleneck.
# Your code here

In [ ]:
# Exercise 5: Benchmark sum(x^2 for x in v) vs sum(v .^ 2) vs a manual loop.
# Which is fastest and why?
# Your code here